# ContractGuard AI — Executed Capstone Evidence

**Program:** SDAIA Academy — Advanced Agentic AI Systems Engineering  
**Cohort/session:** June 2026  
**Project:** Secure Multi-Agent Vendor Contract Audit Platform

This notebook executes and preserves evidence for all six rubric deliverables: real tool
use and named reasoning patterns, framework-managed graph orchestration, role-specialized
agents, security and observability, durable checkpoint/HITL/cloud artifacts, and
professional documentation.


In [1]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)
print('Python:', sys.version.split()[0])


Project root: /mnt/data/contractguard-ai
Python: 3.13.5


## 1. Execute the complete security, retry, HITL, restart, and output-validation demonstration

The runner contains hard assertions. Any missing rubric behavior causes the cell to fail.
It executes a real prompt-injection attack, a safe contract, a high-risk contract with a
simulated tool timeout, a Reflexion re-search loop, a durable human interrupt, a fresh
service restart, human approval, an output-schema revision loop, PII masking, and artifact
storage.


In [2]:
result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'run_capstone_demo.py')],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)


[1/6] Running real prompt-injection attack attempt...
[2/6] Running low-risk happy path...
[3/6] Running tool-failure retry, Reflexion, and HITL pause paths...
[4/6] Closing the process and proving durable restart recovery...
[5/6] Resuming from the human approval interrupt...
[6/6] Writing an evaluator-friendly execution summary...
{
  "prompt_injection_was_blocked": true,
  "tool_failure_retry_fired": true,
  "quality_replan_loop_fired": true,
  "persistent_checkpoint_survived_restart": true,
  "human_pause_resumed": true,
  "output_validation_revision_fired": true,
  "pii_was_masked": true,
  "structured_logs_exist": true,
  "prometheus_metrics_exist": true,
  "cloud_artifact_exists": true
}
All capstone evidence assertions passed.



## 2. Rubric assertion summary

In [3]:
summary = json.loads((PROJECT_ROOT / 'evidence' / 'run_summary.json').read_text())
proof = pd.DataFrame(
    [{'requirement': key.replace('_', ' '), 'passed': value} for key, value in summary['proof'].items()]
)
display(proof)
assert proof['passed'].all()
print('All proof assertions:', bool(proof['passed'].all()))


,requirement,passed
0,prompt injection was blocked,True
1,tool failure retry fired,True
2,quality replan loop fired,True
3,persistent checkpoint survived restart,True
4,human pause resumed,True
5,output validation revision fired,True
6,pii was masked,True
7,structured logs exist,True
8,prometheus metrics exist,True
9,cloud artifact exists,True


All proof assertions: True


## 3. Deliverable 1 — Agentic reasoning and real function tools

The Coordinator implements **Plan-and-Execute**. Every registered function call records a
concise **ReAct** triple: rationale/Thought, Action, and Observation. The Quality Reviewer
implements **Reflexion/self-critique**. The coordinator-to-specialist topology is
**Hierarchical Delegation**.


In [4]:
low = json.loads((PROJECT_ROOT / 'evidence' / '02_low_risk_completed.json').read_text())
tool_rows = []
observations = {o['call_id']: o for o in low['tool_observations']}
for call in low['tool_calls']:
    obs = observations[call['call_id']]
    tool_rows.append({
        'agent': call['agent'],
        'tool': call['tool_name'],
        'rationale': call['rationale'],
        'status': obs['status'],
        'latency_ms': round(obs['latency_ms'], 3),
        'observation': obs['summary'],
    })
display(pd.DataFrame(tool_rows))
print('Named reasoning patterns:', summary['reasoning_patterns'])
print('Shared state keys carried across steps:', len(low.keys()))
assert len(low['tool_calls']) >= 6
assert 'ReAct' in {d.get('pattern') for d in low['decision_trace']}


,agent,tool,rationale,status,latency_ms,observation
0,Document Analyst Agent,read_contract,The document must be converted into trusted te...,success,0.129,"Returned object with keys: source, text, chara..."
1,Document Analyst Agent,extract_contract_clauses,Structured clauses are required so specialist ...,success,0.359,Returned 8 items
2,Policy Research Agent,search_policy_knowledge_base,Need authoritative policy evidence for the dat...,success,0.237,Returned 2 items
3,Policy Research Agent,search_policy_knowledge_base,Need authoritative policy evidence for the gov...,success,0.129,Returned 2 items
4,Policy Research Agent,search_policy_knowledge_base,Need authoritative policy evidence for the lia...,success,0.093,Returned 1 items
5,Policy Research Agent,search_policy_knowledge_base,Need authoritative policy evidence for the pay...,success,0.118,Returned 2 items
6,Policy Research Agent,search_policy_knowledge_base,Need authoritative policy evidence for the sec...,success,0.147,Returned 2 items
7,Policy Research Agent,search_policy_knowledge_base,Need authoritative policy evidence for the ser...,success,0.108,Returned 1 items
8,Policy Research Agent,search_policy_knowledge_base,Need authoritative policy evidence for the ter...,success,0.105,Returned 2 items
9,Security Reviewer Agent,calculate_contract_risk,Risk and value thresholds determine whether au...,success,0.036,"Returned object with keys: risk_score, risk_le..."


Named reasoning patterns: ['Plan-and-Execute', 'ReAct', 'Reflexion/self-critique', 'Hierarchical delegation']
Shared state keys carried across steps: 50


## 4. Deliverable 2 — Genuine graph orchestration with conditions and loops

The graph is built by the `transitions.Machine` framework. Conditions decide branches;
shared state is read and updated by node callbacks; three bounded cycles support tool
retry, Reflexion/re-search, and report revision.


In [5]:
graph = json.loads((PROJECT_ROOT / 'evidence' / 'graph_spec.json').read_text())
print('Framework:', graph['framework'])
print('Node count:', len(graph['nodes']))
print('Loops:')
for loop in graph['loops']:
    print(' -', loop)
edge_frame = pd.DataFrame(graph['edges'])[['trigger', 'source', 'dest', 'conditions', 'before']].fillna('')
display(edge_frame)
assert len(graph['nodes']) >= 10
assert any(edge['source'] == edge['dest'] for edge in graph['edges'])
assert any(edge.get('conditions') for edge in graph['edges'])


Framework: transitions finite-state machine
Node count: 16
Loops:
 - researching -> researching (tool retry)
 - quality_reviewed -> researching (Reflexion/re-plan)
 - output_validated -> reporting (schema-revision loop)


,trigger,source,dest,conditions,before
0,advance,received,guardrailed,,
1,advance,guardrailed,blocked,_input_is_blocked,
2,advance,guardrailed,planned,_input_is_safe,
3,advance,planned,ingested,,
4,advance,ingested,researching,,
5,advance,researching,researching,_policy_error_can_retry,_record_policy_retry
6,advance,researching,failed,_policy_error_exhausted,_mark_retry_exhausted
7,advance,researching,analyzed,_policy_search_succeeded,
8,advance,analyzed,quality_reviewed,,
9,advance,quality_reviewed,researching,_quality_needs_retry,_record_quality_retry


## 5. Failure paths — actual tool retry and Reflexion re-search

The first policy search deliberately raises a simulated timeout. The failed
`ToolObservation` is retained, the conditional self-loop fires, and the next attempt
succeeds. A separate quality critique routes back to research.


In [6]:
paused = json.loads((PROJECT_ROOT / 'evidence' / '03_high_risk_paused_for_human.json').read_text())
failed_tools = [o for o in paused['tool_observations'] if o['status'] == 'error']
print('Failed tool observations:', json.dumps(failed_tools, indent=2))
print('Policy retry count:', paused['policy_retry_count'])
print('Quality re-plan count:', paused['quality_retry_count'])
print('Research node visits:', paused['node_history'].count('researching'))
print('Node path:', ' -> '.join(paused['node_history']))
assert failed_tools
assert paused['policy_retry_count'] >= 1
assert paused['quality_retry_count'] >= 1
assert paused['node_history'].count('researching') >= 3


Failed tool observations: [
  {
    "call_id": "3cf44338-792f-413f-bf4a-b96ea971cfe2",
    "tool_name": "search_policy_knowledge_base",
    "status": "error",
    "summary": "ConnectionError: Simulated primary policy-index timeout for retry-path evidence",
    "output": null,
    "latency_ms": 0.07054799993966299,
    "timestamp": "2026-08-12T20:21:44.512403+00:00"
  }
]
Policy retry count: 1
Quality re-plan count: 1
Research node visits: 3
Node path: received -> guardrailed -> planned -> ingested -> researching -> researching -> analyzed -> quality_reviewed -> researching -> analyzed -> quality_reviewed -> security_reviewed -> awaiting_approval


## 6. Deliverable 3 — Multi-agent role specialization and structured communication

These are separate agent objects/classes, not personas concatenated into one prompt.
Messages contain sender, recipient, message type, content, payload, and timestamp.


In [7]:
messages = pd.DataFrame(paused['agent_messages'])
agent_summary = messages.groupby('sender').agg(
    messages=('sender', 'size'),
    recipients=('recipient', lambda values: ', '.join(sorted(set(values))))
).reset_index()
display(agent_summary)
display(messages[['sender', 'recipient', 'message_type', 'content']].tail(12))
print('Coordination strategy:', summary['coordination_strategy'])
assert messages['sender'].nunique() >= 7
assert {'sender', 'recipient', 'message_type', 'payload'}.issubset(messages.columns)


,sender,messages,recipients
0,Compliance Analyst Agent,2,Quality Reviewer Agent
1,Coordinator Agent,1,Document Analyst Agent
2,Document Analyst Agent,1,Policy Research Agent
3,Input Security Agent,1,Coordinator Agent
4,Policy Research Agent,3,"Compliance Analyst Agent, Coordinator Agent"
5,Quality Reviewer Agent,2,Coordinator Agent
6,Security Reviewer Agent,1,Coordinator Agent


,sender,recipient,message_type,content
0,Input Security Agent,Coordinator Agent,security,Input guardrail passed; execution may proceed.
1,Coordinator Agent,Document Analyst Agent,plan,Plan-and-Execute plan created with seven bound...
2,Document Analyst Agent,Policy Research Agent,handoff,Extracted 9 clauses and contract metadata.
3,Policy Research Agent,Coordinator Agent,status,Policy tool failed; requesting graph retry.
4,Policy Research Agent,Compliance Analyst Agent,result,Retrieved 12 policy evidence records.
5,Compliance Analyst Agent,Quality Reviewer Agent,result,Generated 7 compliance findings from contract ...
6,Quality Reviewer Agent,Coordinator Agent,critique,Evidence coverage score is 50%; re-planning ta...
7,Policy Research Agent,Compliance Analyst Agent,result,Retrieved 14 policy evidence records.
8,Compliance Analyst Agent,Quality Reviewer Agent,result,Generated 7 compliance findings from contract ...
9,Quality Reviewer Agent,Coordinator Agent,critique,Evidence coverage score is 100%; quality gate ...


Coordination strategy: Centralized Coordinator Agent with structured specialist handoffs


## 7. Deliverable 4 — Security guardrails and structured observability

The malicious uploaded contract is inspected before the tool registry is reachable. The
output guardrail masks PII and validates a strict Pydantic schema. Monitoring is JSONL +
Prometheus, not print statements.


In [8]:
blocked = json.loads((PROJECT_ROOT / 'evidence' / '01_prompt_injection_blocked.json').read_text())
print('Attack terminal status:', blocked['status'])
print('Detected reason:', blocked['blocked_reason'])
print('Tool calls after block:', len(blocked['tool_calls']))
print('Blocked path:', ' -> '.join(blocked['node_history']))
assert blocked['status'] == 'blocked'
assert len(blocked['tool_calls']) == 0


Attack terminal status: blocked
Detected reason: ignore previous instructions; reveal system prompt; system prompt extraction
Tool calls after block: 0
Blocked path: received -> guardrailed -> blocked


In [9]:
log_path = PROJECT_ROOT / 'evidence' / 'execution_log.jsonl'
logs = [json.loads(line) for line in log_path.read_text().splitlines() if line.strip()]
log_frame = pd.DataFrame(logs)
print('Structured log events:', len(log_frame))
print('Event types:', sorted(log_frame['event'].dropna().unique()))
display(log_frame[['timestamp', 'event', 'thread_id', 'node', 'tool', 'latency_ms']].tail(15).fillna(''))

metrics_text = (PROJECT_ROOT / 'evidence' / 'metrics_before_restart.prom').read_text()
metric_names = sorted({line.split('{', 1)[0].split(' ', 1)[0] for line in metrics_text.splitlines() if line and not line.startswith('#')})
print('Prometheus metric series (sample):', metric_names[:20])
assert 'tool_call_failed' in set(log_frame['event'])
assert 'guardrail_blocked' in set(log_frame['event'])
assert 'human_interrupt' in set(log_frame['event'])
assert 'contractguard_tool_calls_total' in metrics_text


Structured log events: 195
Event types: ['agent_message', 'checkpoint_saved', 'guardrail_blocked', 'human_decision', 'human_interrupt', 'metrics_exported', 'node_completed', 'node_started', 'tool_call_completed', 'tool_call_failed', 'tool_call_started', 'workflow_retry', 'workflow_started', 'workflow_terminal']


,timestamp,event,thread_id,node,tool,latency_ms
180,2026-08-12T20:21:44.716674+00:00,node_started,capstone-high-risk,output_validated,,
181,2026-08-12T20:21:44.718832+00:00,agent_message,capstone-high-risk,,,
182,2026-08-12T20:21:44.718883+00:00,node_completed,capstone-high-risk,output_validated,,2.195
183,2026-08-12T20:21:44.721531+00:00,checkpoint_saved,capstone-high-risk,output_validated,,
184,2026-08-12T20:21:44.721634+00:00,node_started,capstone-high-risk,persisting,,
185,2026-08-12T20:21:44.721749+00:00,tool_call_started,capstone-high-risk,,store_report_artifact,
186,2026-08-12T20:21:44.722184+00:00,tool_call_completed,capstone-high-risk,,store_report_artifact,0.372
187,2026-08-12T20:21:44.722330+00:00,agent_message,capstone-high-risk,,,
188,2026-08-12T20:21:44.722395+00:00,node_completed,capstone-high-risk,persisting,,0.725
189,2026-08-12T20:21:44.725771+00:00,checkpoint_saved,capstone-high-risk,persisting,,


Prometheus metric series (sample): ['contractguard_active_workflows', 'contractguard_estimated_llm_cost_usd_created', 'contractguard_estimated_llm_cost_usd_total', 'contractguard_guardrail_blocks_created', 'contractguard_guardrail_blocks_total', 'contractguard_human_interrupts_created', 'contractguard_human_interrupts_total', 'contractguard_node_latency_seconds_bucket', 'contractguard_node_latency_seconds_count', 'contractguard_node_latency_seconds_created', 'contractguard_node_latency_seconds_sum', 'contractguard_node_runs_created', 'contractguard_node_runs_total', 'contractguard_retries_created', 'contractguard_retries_total', 'contractguard_tool_calls_created', 'contractguard_tool_calls_total', 'contractguard_tool_latency_seconds_bucket', 'contractguard_tool_latency_seconds_count', 'contractguard_tool_latency_seconds_created']


## 8. Deliverable 5 — Persistent checkpoint, real HITL pause/resume, and cloud artifact

A high-risk contract stops at `awaiting_approval`. The first service object is closed.
A fresh service object opens the same SQLite database, reloads the thread, applies a human
decision, and continues from the paused node.


In [10]:
loaded = json.loads((PROJECT_ROOT / 'evidence' / '04_checkpoint_loaded_after_restart.json').read_text())
final = json.loads((PROJECT_ROOT / 'evidence' / '05_high_risk_resumed_and_completed.json').read_text())
print('Node loaded after restart:', loaded['node'])
print('Interrupt payload:', json.dumps(loaded['state']['interrupt_payload'], indent=2))
print('Final status:', final['status'])
print('Human decision:', final['approval_status'], '-', final['approver'])
print('Output revision count:', final['report_revision_count'])
print('PII redactions:', final['pii_redactions'])
print('Artifact URI:', final['artifact_uri'])
assert loaded['node'] == 'awaiting_approval'
assert final['status'] == 'completed'
assert final['approval_status'] == 'approved'
assert final['report_revision_count'] >= 1
assert final['pii_redactions'] >= 3


Node loaded after restart: awaiting_approval
Interrupt payload: {
  "type": "human_approval_required",
  "thread_id": "capstone-high-risk",
  "risk_score": 99,
  "risk_level": "CRITICAL",
  "contract_value_sar": 750000.0,
  "finding_count": 7,
  "instruction": "Resume with decision=approve or decision=reject and an approver identity."
}
Final status: completed
Human decision: approved - Capstone Human Reviewer
Output revision count: 1
PII redactions: 3
Artifact URI: file:///mnt/data/contractguard-ai/evidence/artifacts/capstone-high-risk/compliance_report.md


In [11]:
cloud_files = ['Dockerfile', 'docker-compose.yml', 'deploy/prometheus.yml', 'src/contractguard/api.py']
cloud_evidence = []
for relative in cloud_files:
    path = PROJECT_ROOT / relative
    cloud_evidence.append({'artifact': relative, 'exists': path.exists(), 'bytes': path.stat().st_size if path.exists() else 0})
display(pd.DataFrame(cloud_evidence))
assert all(item['exists'] for item in cloud_evidence)


,artifact,exists,bytes
0,Dockerfile,True,767
1,docker-compose.yml,True,1703
2,deploy/prometheus.yml,True,188
3,src/contractguard/api.py,True,2797


## 9. Final masked compliance report

In [12]:
report_path = Path(final['artifact_uri'].removeprefix('file://'))
report_text = report_path.read_text()
print(report_text[:6000])
assert '[REDACTED_EMAIL]' in report_text
assert '[REDACTED_PHONE]' in report_text
assert '[REDACTED_NATIONAL_ID]' in report_text
assert 'nora.alqahtani@example.com' not in report_text


# ContractGuard AI Compliance Report

**Report ID:** CGA-capstone-high-risk  
**Thread ID:** capstone-high-risk  
**Generated:** 2026-08-12T20:21:44.713989+00:00  
**Vendor:** Falcon Cloud International LLC  
**Contract value:** SAR 750,000.00  
**Risk:** CRITICAL (99/100)  
**Approval:** approved  

## Executive Summary

The multi-agent audit identified 7 compliance finding(s) and assigned a CRITICAL risk rating of 99/100. The result was produced from clause extraction, policy retrieval, specialist review, security routing, and output validation rather than a single prompt.

## Compliance Findings

| ID | Severity | Topic | Finding | Policy reference |
|---|---|---|---|---|
| F-01 | HIGH | data protection | Unrestricted cross-border data transfer | Data Protection Policy — Saudi Data Residency |
| F-02 | HIGH | data protection | Subprocessor changes lack customer consent | Data Protection Policy — Saudi Data Residency |
| F-03 | HIGH | data protection | Breach notification is missing 

## 10. Automated tests

The test suite covers direct/indirect injection, PII masking, real tool search, low-risk
completion, graph retry, restart persistence, HITL resume, output revision, artifact
storage, and FastAPI endpoints.


In [13]:
tests = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=True,
    env={**os.environ, 'PYTHONPATH': str(PROJECT_ROOT / 'src')},
)
print(tests.stdout)
(PROJECT_ROOT / 'evidence' / 'pytest_results.txt').write_text(tests.stdout + tests.stderr)


........                                                                 [100%]



161

## 11. Documentation and submission completeness

- Professional README with setup, API keys, expected outputs, deployment, evidence index,
  training attribution, and SDAIA Academy link.
- Technical architecture using nodes, edges, state, agents, tools, conditions, loops, and
  checkpointers.
- Rubric traceability, security model, API reference, tests, `.gitignore`, Docker/Compose,
  CI workflow, and retained third-party license.
- Executed notebook and captured JSON/log/metric/report evidence.

The only external submission step is pushing this prepared Git repository to the
trainee's chosen GitHub repository.


In [14]:
required = [
    'README.md', '.gitignore', 'docs/architecture.md', 'docs/rubric_traceability.md',
    'docs/security.md', 'docs/api.md', 'Dockerfile', 'docker-compose.yml',
    '.github/workflows/ci.yml', 'THIRD_PARTY_NOTICES.md',
]
rows = [{'file': item, 'exists': (PROJECT_ROOT / item).exists()} for item in required]
display(pd.DataFrame(rows))
assert all(row['exists'] for row in rows)
print('Executed capstone notebook completed successfully.')


,file,exists
0,README.md,True
1,.gitignore,True
2,docs/architecture.md,True
3,docs/rubric_traceability.md,True
4,docs/security.md,True
5,docs/api.md,True
6,Dockerfile,True
7,docker-compose.yml,True
8,.github/workflows/ci.yml,True
9,THIRD_PARTY_NOTICES.md,True


Executed capstone notebook completed successfully.
